# 브랜드 리뷰 자동 수집 & NPS 정량화 파이프라인

**흐름:** 링크 입력 → 리뷰 자동 수집(robots.txt 자동 확인, 마지막 페이지까지 전부 순회) → 점수 산정(실제 별점 우선, 없으면 감성 추정) → NPS 계산/표 → CSV 저장 → 그래프

| 단계 | 내용 |
|---|---|
| 1 | 분석할 링크(URL)와 브랜드명 입력 |
| 2 | 그 링크에서 리뷰 텍스트(+ 있다면 실제 별점) 자동 수집 — 리뷰 수와 무관하게 마지막 페이지까지 순회 |
| 3 | 리뷰마다 1~5점 점수 + 핵심 키워드 자동 부여 (기준은 아래 4단계 참고) |
| 4 | 추천고객/중립고객/비추천고객 비율 계산 → 소셜 NPS 점수 산출 |
| 5 | 결과를 CSV로 저장 |
| 6 | 막대그래프·파이차트로 시각화 |

**예시 링크로 검증한 내용** (`https://www.antursolais.com/board/product/list.html?board_no=4`, 카페24 기반 쇼핑몰의 리뷰 게시판):
- 정적 HTML이라 브라우저 없이 `requests`만으로 수집됩니다.
- 목록에 고객이 남긴 **실제 별점(1~5)**이 그대로 노출되어 있어, 감성을 추정할 필요 없이 그 값을 점수로 그대로 씁니다(추정보다 훨씬 정확).
- **자동 수집기는 이제 1페이지에서 멈추지 않습니다.** 2, 3, 4…페이지로 계속 순회하면서 "더 이상 새 리뷰가 없음"(마지막 페이지)을 만날 때까지 전부 모읍니다. 단, 매 페이지 요청 전 그 페이지의 `robots.txt` 허용 여부를 확인해서, 차단된 페이지를 만나면 그 지점에서 멈춥니다(사이트가 스스로 정한 크롤링 정책이라 우회하지 않습니다).
- 이 예시 사이트는 `robots.txt`가 `&page=` 파라미터가 붙은 URL 자체를 막아둬서(직접 확인함), 자동 수집은 여전히 1페이지(약 25건)까지만 가능합니다 — 이건 코드의 한계가 아니라 사이트가 명시적으로 막아둔 지점입니다. 더 필요하면 카페24 관리자 페이지의 게시판 내보내기 기능이나, (상세 페이지에 "스마트스토어에서 등록된 구매후기"라고 표시된 걸 보면 이 리뷰들이 네이버 스마트스토어와 연동되어 있으므로) 스마트스토어 판매자센터의 리뷰 다운로드를 받아 `COLLECT_MODE = "manual"`로 불러오는 걸 권장합니다.
- 참고로 이 사이트의 리뷰 게시판 자체는 구글에 색인돼 있어(`site:antursolais.com 후기` 등으로 검색하면 페이지가 나옵니다) 실존하는 리뷰가 맞습니다. 개별 리뷰 문장 하나하나가 검색 결과에 그대로 잡히지는 않는데, 이는 소규모 쇼핑몰 게시판 특성상 흔한 일이라 "가짜라서 안 나온다"는 신호는 아닙니다.
- `board/product/list.html?board_no=` 패턴은 카페24를 쓰는 다른 쇼핑몰에도 흔히 쓰이는 구조라, 도메인과 `board_no`만 바꿔도 대체로 재사용됩니다. **robots.txt가 페이지네이션을 막지 않는 사이트라면, 이제 리뷰 수와 무관하게 마지막 페이지까지 자동으로 전부 수집됩니다.**
- 이 패턴이 아닌 사이트를 넣으면 일반 휴리스틱 수집기로 자동 전환되며, 이쪽도 동일하게 페이지네이션을 끝까지 순회합니다(다만 JS로 나중에 로딩되는 사이트는 잡지 못합니다 — 아래 한계 참고).


## 1. 라이브러리 설치 및 임포트

In [ ]:
# 필요한 패키지 설치 (최초 1회만 실행하면 됩니다)
!pip install -q requests beautifulsoup4 lxml pandas matplotlib koreanize-matplotlib google-genai tqdm

In [ ]:
import re
import json
import time
from getpass import getpass

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 그래프에서 한글이 깨지지 않도록 폰트 자동 설정
from tqdm import tqdm

pd.set_option("display.max_colwidth", 80)
print("준비 완료")

## 1-1. 그래프/표 스타일 설정 — 나눔고딕 · 큰 글씨 · 볼드체

이후 모든 그래프와 표에 공통으로 적용됩니다. 나눔고딕이 시스템에 없으면 자동 설치를 시도하고(Colab/Linux 기준),
설치가 안 되는 환경(오프라인 등)에서는 `koreanize_matplotlib`이 잡아준 기본 한글 폰트로 자동 대체됩니다.


In [ ]:
import matplotlib.font_manager as fm
import matplotlib as mpl


def setup_nanumgothic_style():
    """나눔고딕 폰트를 찾아서(없으면 설치 시도) matplotlib 기본 폰트로 지정하고,
    전체 그래프의 글자 크기를 키우고 볼드체를 기본값으로 설정합니다."""
    font_path = None
    for f in fm.fontManager.ttflist:
        if "Nanum" in f.name and "Gothic" in f.name:
            font_path = f.fname
            break

    if font_path is None:
        import subprocess
        try:
            subprocess.run(["apt-get", "-qq", "update"], check=True,
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=True,
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            candidate = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
            fm.fontManager.addfont(candidate)
            font_path = candidate
        except Exception:
            pass  # 오프라인 환경 등에서 설치가 안 되면 아래에서 대체 폰트로 넘어갑니다.

    if font_path:
        font_name = fm.FontProperties(fname=font_path).get_name()
    else:
        current = plt.rcParams["font.family"]
        font_name = current[0] if isinstance(current, list) else current
        print(f"\u26a0\ufe0f 나눔고딕 자동 설치에 실패해 현재 환경의 기본 한글 폰트로 대체합니다: {font_name}")

    mpl.rcParams["font.family"] = font_name
    mpl.rcParams["axes.unicode_minus"] = False
    mpl.rcParams["font.size"] = 14
    mpl.rcParams["font.weight"] = "bold"
    mpl.rcParams["axes.titlesize"] = 20
    mpl.rcParams["axes.titleweight"] = "bold"
    mpl.rcParams["axes.labelsize"] = 15
    mpl.rcParams["axes.labelweight"] = "bold"
    mpl.rcParams["xtick.labelsize"] = 13
    mpl.rcParams["ytick.labelsize"] = 13
    mpl.rcParams["legend.fontsize"] = 13
    mpl.rcParams["figure.titlesize"] = 22
    mpl.rcParams["figure.titleweight"] = "bold"
    print(f"\u2705 그래프 스타일 설정 완료 \u2192 폰트: {font_name} (볼드체, 큰 글씨 기본 적용)")
    return font_name


NANUM_FONT = setup_nanumgothic_style()


## 2. 설정 — 분석할 링크와 브랜드명

여기 세 줄만 바꿔서 다시 실행하면 다른 브랜드/링크도 바로 분석할 수 있습니다.

In [ ]:
# ================================================================
# 여기에 분석하고 싶은 리뷰 페이지 링크와 브랜드명을 입력하세요
# ================================================================
TARGET_URL = "https://www.antursolais.com/board/product/list.html?board_no=4"
BRAND_NAME = "앙투어솔레"     # 표/그래프 제목에 표시될 이름

# 리뷰 수집 방식
#   "auto"   : 아래 자동 수집 함수로 시도 (정적 HTML 페이지에 적합, 마지막 페이지까지 전부 순회)
#   "manual" : REVIEWS_CSV_PATH 의 CSV에서 리뷰를 불러옴 (자동 수집이 막힌 사이트용)
COLLECT_MODE = "auto"
REVIEWS_CSV_PATH = "reviews.csv"   # COLLECT_MODE="manual"일 때 사용, '리뷰' 컬럼 필요

# 자동 수집 시 최대 몇 페이지까지 순회할지 (무한 루프 방지용 안전장치일 뿐, 보통 그 전에 멈춥니다)
MAX_PAGES = 500

# 리뷰를 최대 몇 건까지만 모을지. 정수를 넣으면 그 건수에 도달하는 즉시 수집을 멈춥니다.
# 전체 다 모으고 싶으면 None으로 두세요.
MAX_ITEMS = None

# 감성 분류 방식
#   "keyword" : 키워드 사전 기반 규칙 스코어링 (API 키 불필요, 무료, 즉시 실행)
#   "llm"     : 무료 LLM API(Google Gemini) 기반 정밀 분류 (문맥·부정어까지 이해, API 키만 있으면 무료)
SENTIMENT_METHOD = "llm"

# llm 모드일 때, 한 번의 API 호출에 리뷰를 몇 건씩 묶어서 보낼지.
# 크게 잡을수록 호출 횟수가 줄어 훨씬 빨라지지만, 한 번에 받아야 하는 응답이 길어져
# 모델의 출력 길이 제한에 걸려 잘릴 위험도 커집니다.
# → 응답이 잘리거나 파싱에 실패하면(로그에 "배치 처리 실패"가 자주 뜨면) 이 값을 낮추세요.
#   (배치가 실패해도 자동으로 절반씩 쪼개 재시도하므로, 일단 크게 잡아도 안전하게 동작합니다)
LLM_BATCH_SIZE = 50

print(f"TARGET_URL = {TARGET_URL!r}")
print(f"BRAND_NAME = {BRAND_NAME!r}")
print(f"COLLECT_MODE = {COLLECT_MODE!r}, MAX_PAGES = {MAX_PAGES}, MAX_ITEMS = {MAX_ITEMS}, SENTIMENT_METHOD = {SENTIMENT_METHOD!r}, LLM_BATCH_SIZE = {LLM_BATCH_SIZE}")


## 3. 리뷰 자동 수집

세 단계로 동작합니다.

1. **robots.txt 자동 확인** — 페이지를 요청하기 전에 매번 대상 URL의 `robots.txt`를 읽어, 자동 수집이 허용되는지 확인합니다. 차단된 URL이면 요청을 보내지 않고 건너뜁니다.
2. **패턴 감지 수집** — 먼저 카페24 게시판형 리뷰 목록(`.../board/xxx/list.html?board_no=...`) 패턴을 시도합니다. 이 구조는 리뷰 텍스트뿐 아니라 **실제 별점(1~5)**까지 HTML에 그대로 들어 있어, 감성 추정 없이 진짜 점수를 바로 씁니다. 이 패턴이 아니면, class/id에 `review`/`comment`/`후기`/`리뷰`/`댓글` 같은 단어가 포함된 블록을 후보로 잡는 일반 휴리스틱으로 자동 전환됩니다(이 경우 점수는 4단계의 키워드/LLM 감성 추정으로 매깁니다).
3. **끝까지 페이지네이션 순회** — 1페이지를 모은 뒤 2, 3, 4…페이지로 계속 넘어가면서, "새로 나온 리뷰가 없음"(마지막 페이지 도달) 또는 "robots.txt가 다음 페이지를 차단함"을 만날 때까지 반복합니다. 즉 리뷰 수가 몇 건이든 사이트가 허용하는 한 전부 수집됩니다 — `MAX_PAGES`는 무한 루프 방지용 안전장치일 뿐, 보통은 그 전에 자연스럽게 멈춥니다.

**한계**:
- 인스타그램, 앱스토어, 지도 서비스처럼 리뷰를 JavaScript로 나중에 불러오는 사이트(SPA)는 이 방식으로 수집되지 않습니다. 이런 경우 아래 "3-1. Selenium 기반 수집(선택)"을 사용하거나, `COLLECT_MODE = "manual"`로 바꿔 CSV로 직접 업로드하세요.
- 사이트가 `robots.txt`로 특정 페이지부터 자동 수집을 막아둔 경우, 그 지점에서 의도적으로 멈춥니다(우회하지 않습니다). 이럴 땐 사이트가 공식 제공하는 내보내기/다운로드 기능(쇼핑몰 관리자 페이지, 스마트스토어 판매자센터 등)으로 받은 리뷰를 `manual` 모드로 불러오는 걸 권장합니다.


In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

REVIEW_HINT_PATTERNS = ["review", "comment", "reply", "feedback", "후기", "리뷰", "댓글"]


def fetch_html(url: str, timeout: int = 10) -> bytes:
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return resp.content  # 디코딩하지 않은 원본 바이트 — BeautifulSoup이 <meta charset>/BOM 등을 보고
                          # requests의 apparent_encoding(통계적 추측)보다 훨씬 안정적으로 인코딩을 감지합니다.
                          # (requests가 인코딩을 잘못 추측하면 한글/이모지가 깨져 보이는 원인이 됩니다)


def is_allowed_by_robots(url: str, user_agent: str = "*") -> bool:
    # 대상 사이트의 robots.txt를 읽어 이 URL의 자동 수집이 허용되는지 확인
    from urllib.parse import urlparse
    from urllib.robotparser import RobotFileParser

    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
    except Exception:
        return True  # robots.txt를 읽을 수 없으면 기본 허용으로 간주 (필요시 직접 확인하세요)
    return rp.can_fetch(user_agent, url)


def build_page_url(url: str, page: int) -> str:
    # 원래 URL의 다른 쿼리 파라미터(board_no 등)는 그대로 둔 채 page 파라미터만 바꿔치기
    from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

    parts = urlsplit(url)
    query = dict(parse_qsl(parts.query))
    query["page"] = str(page)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(query), parts.fragment))


def extract_rating_from_row(row_tag):
    # 같은 행(<tr>) 안의 별점 아이콘에서 1~5 숫자를 추출 (카페24 리뷰 게시판 패턴)
    if row_tag is None:
        return None
    img = row_tag.find("img", src=re.compile(r"rating\d"))
    if img is None:
        img = row_tag.find("img", alt=re.compile(r"\d\s*점"))
    if img is None:
        return None
    m = re.search(r"rating(\d)", img.get("src", "")) or re.search(r"(\d)\s*점", img.get("alt", ""))
    return int(m.group(1)) if m else None


def extract_cafe24_review_rows(soup: BeautifulSoup) -> list[dict]:
    # 카페24 게시판형 리뷰 목록 패턴 전용 추출기 — 텍스트 + 실제 별점을 함께 반환
    items, seen = [], set()
    for a in soup.select('a[href*="/article/review/"]'):
        href = a.get("href", "")
        text = a.get_text(strip=True)
        if not text or href in seen:
            continue
        seen.add(href)
        rating = extract_rating_from_row(a.find_parent("tr"))
        items.append({"리뷰": text, "별점": rating, "링크": href})
    return items


def guess_review_elements(soup: BeautifulSoup) -> list[str]:
    # class/id 이름에 리뷰 관련 단어가 포함된 블록의 텍스트를 후보로 추출 (카페24 패턴이 아닐 때의 대안)
    candidates = []
    for tag in soup.find_all(True):
        attrs = " ".join(tag.get("class", []) + [tag.get("id", "")]).lower()
        if any(p in attrs for p in REVIEW_HINT_PATTERNS):
            text = tag.get_text(strip=True)
            if 5 <= len(text) <= 500:   # 너무 짧거나(버튼 등) 너무 긴(전체 페이지) 텍스트 제외
                candidates.append(text)
    seen, reviews = set(), []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            reviews.append(c)
    return reviews


def collect_reviews_auto(url: str, max_pages: int = 500, max_items: int | None = None) -> list[dict]:
    """1페이지에서 멈추지 않고, 마지막 페이지까지(또는 max_items에 도달할 때까지) 순회하며 수집합니다.
    페이지를 처리할 때마다 몇 건을 모았는지, 누적 몇 건인지를 실시간으로 출력합니다.

    멈추는 조건:
      1) max_items를 지정했고 누적 수집 건수가 거기 도달함 → 그 즉시 멈춤
      2) 다음 페이지에 새로 나오는 리뷰가 없음 → 마지막 페이지에 도달한 것으로 판단
      3) robots.txt가 다음 페이지의 자동 수집을 차단함 → 사이트가 스스로 정한 정책이므로 그 지점에서 멈추고
         우회하지 않습니다 (이럴 땐 관리자 페이지 내보내기 등 공식 경로로 CSV를 받아 manual 모드로 불러오세요).
    """
    if not is_allowed_by_robots(url):
        print(f"⛔ robots.txt가 이 URL의 자동 수집을 차단하고 있어 건너뜁니다: {url}")
        print("   (목록 URL은 허용하면서 페이지네이션(&page=)만 막아둔 사이트가 흔합니다 — TARGET_URL을 파라미터 없는 기본 목록 링크로 바꿔보세요)")
        return []

    collected: list[dict] = []
    seen_keys = set()
    mode = None  # "cafe24" 또는 "heuristic" — 1페이지에서 감지된 패턴을 이후 페이지에도 동일하게 적용
    last_page_checked = 0

    for page in range(1, max_pages + 1):
        page_url = url if page == 1 else build_page_url(url, page)

        if page > 1 and not is_allowed_by_robots(page_url):
            print(f"⛔ robots.txt가 {page}페이지부터는 자동 수집을 막고 있어 여기서 멈춥니다: {page_url}")
            print(f"   사이트가 명시적으로 막은 지점이라 더 진행하지 않습니다. 최종 수집: {len(collected)}건 (확인한 페이지: {last_page_checked}개).")
            print("   더 필요하면 사이트의 공식 내보내기/다운로드 기능으로 받은 CSV를 COLLECT_MODE='manual'로 불러오세요.")
            break

        print(f"[{page}페이지] 요청 중... ({page_url})")
        try:
            soup = BeautifulSoup(fetch_html(page_url), "lxml")
        except Exception as e:
            print(f"  ⚠️ {page}페이지 요청 실패, 여기서 멈춥니다: {e}")
            break

        last_page_checked = page

        if mode in (None, "cafe24"):
            items = extract_cafe24_review_rows(soup)
            if items:
                mode = "cafe24"
                new_items = [it for it in items if it["링크"] not in seen_keys]
                if not new_items:
                    print(f"  ↳ {page}페이지: 새 리뷰 없음 → 마지막 페이지로 판단, 수집 종료")
                    break
                seen_keys.update(it["링크"] for it in new_items)
                collected.extend(new_items)
                print(f"  ↳ {page}페이지: {len(new_items)}건 수집 (누적 {len(collected)}건, 실제 별점 포함)")
                if max_items is not None and len(collected) >= max_items:
                    collected = collected[:max_items]
                    print(f"  ↳ 목표 건수({max_items}건) 도달, 수집 종료")
                    break
                time.sleep(0.4)  # 서버에 과도한 요청을 보내지 않도록 페이지 사이 짧게 대기
                continue
            elif mode == "cafe24":
                print(f"  ↳ {page}페이지: 리뷰 없음 → 마지막 페이지로 판단, 수집 종료")
                break

        texts = guess_review_elements(soup)
        new_texts = [t for t in texts if t not in seen_keys]
        if not new_texts:
            print(f"  ↳ {page}페이지: 새 리뷰 없음 → 마지막 페이지로 판단, 수집 종료")
            break
        mode = "heuristic"
        seen_keys.update(new_texts)
        collected.extend({"리뷰": t, "별점": None, "링크": None} for t in new_texts)
        print(f"  ↳ {page}페이지: {len(new_texts)}건 수집 (누적 {len(collected)}건, 휴리스틱 추출)")
        if max_items is not None and len(collected) >= max_items:
            collected = collected[:max_items]
            print(f"  ↳ 목표 건수({max_items}건) 도달, 수집 종료")
            break
        time.sleep(0.4)

    if not collected:
        print("⚠️ 자동으로 추출된 리뷰가 없습니다. 다음 중 하나일 수 있습니다:")
        print("  1) 리뷰를 JavaScript로 나중에 불러오는 페이지 (인스타그램, 앱스토어, 지도 리뷰 등)")
        print("  2) 로그인/캡차 등으로 스크래핑이 차단된 사이트")
        print("  3) 리뷰 영역의 class/id에 review/comment 같은 키워드가 없는 사이트")
        print("→ 아래 'Selenium 기반 수집' 셀을 쓰거나, COLLECT_MODE='manual'로 CSV를 직접 불러오세요.")
        return []

    label = "카페24 게시판형 리뷰 패턴 감지 — 실제 별점 포함 " if mode == "cafe24" else ""
    print(f"\n✅ {label}최종 총 {len(collected)}건 수집 완료 (확인한 페이지: {last_page_checked}개)")
    return collected


### 3-1. Selenium 기반 수집 (선택 — JavaScript 렌더링 사이트용)

주석을 해제해서 사용하세요. 사이트마다 리뷰가 담긴 CSS 선택자(`CSS_SELECTOR`)가 다르므로, 브라우저 개발자 도구(F12 → Elements)로 리뷰 텍스트를 감싸는 태그의 class를 직접 확인해서 넣어야 합니다.

In [ ]:
# !pip install -q selenium webdriver-manager
#
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.chrome.service import Service
# from webdriver_manager.chrome import ChromeDriverManager
#
# def collect_reviews_selenium(url: str, css_selector: str, scroll_count: int = 5) -> list[str]:
#     options = Options()
#     options.add_argument("--headless=new")
#     driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
#     driver.get(url)
#     for _ in range(scroll_count):          # 무한 스크롤 페이지 대응
#         driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
#         time.sleep(1.5)
#     elements = driver.find_elements(By.CSS_SELECTOR, css_selector)
#     reviews = [e.text.strip() for e in elements if e.text.strip()]
#     driver.quit()
#     return reviews
#
# reviews = collect_reviews_selenium(TARGET_URL, css_selector=".review-text")  # 예시 선택자 — 실제 사이트에 맞게 수정

### 실행 — 리뷰 수집

In [ ]:
if COLLECT_MODE == "auto":
    reviews = collect_reviews_auto(TARGET_URL, max_pages=MAX_PAGES, max_items=MAX_ITEMS)
elif COLLECT_MODE == "manual":
    df_manual = pd.read_csv(REVIEWS_CSV_PATH)
    reviews = [{"리뷰": t, "별점": None, "링크": None} for t in df_manual["리뷰"].dropna().astype(str)]
    if MAX_ITEMS is not None:
        reviews = reviews[:MAX_ITEMS]
else:
    raise ValueError("COLLECT_MODE는 'auto' 또는 'manual' 이어야 합니다.")

print(f"수집된 리뷰 개수: {len(reviews)}건")
pd.DataFrame(reviews).head(5)


In [ ]:
# ================================================================
# 원본 수집 데이터 즉시 백업 — 점수 매기기 전 상태를 그대로 CSV로 저장
# (노트북 화면에는 위에서 head(5)로 5건만 미리보기로 보이지만,
#  reviews 변수 자체에는 수집된 전체 건수가 들어있습니다)
# ================================================================
raw_csv_path = f"{BRAND_NAME.replace(' ', '_')}_원본수집리뷰.csv"
pd.DataFrame(reviews).to_csv(raw_csv_path, index=False, encoding="utf-8-sig")
print(f"저장 완료 → {raw_csv_path} (원본 리뷰 {len(reviews)}건, 점수 매기기 전 상태)")


## 4. 점수 산정 — 정량 평가 기준

**우선순위: 실제 별점 > 감성 추정.** 3단계에서 카페24 패턴으로 수집된 리뷰는 이미 고객이 남긴 실제 별점(1~5)을 갖고 있으므로, 그 값을 그대로 점수로 사용합니다(가장 정확함). 별점이 없는 리뷰(일반 휴리스틱 수집분, 또는 `manual` 업로드분)만 아래 두 방법 중 하나로 점수를 추정합니다. "핵심 키워드"는 별점 유무와 무관하게 모든 리뷰에서 항상 추출합니다(인사이트용).

**방법 A. 키워드 사전 기반 (`keyword`)** — 완전 무료, 즉시 실행, 결과가 결정적(deterministic)입니다. 단, "불친절하지 않았어요" 같은 부정어·반어법은 잘 구분하지 못합니다. 아래 사전은 예시로 넣어드린 치즈/델리 식품 브랜드(앙투어솔레) 리뷰에서 실제로 등장한 표현을 반영했습니다 — 다른 브랜드/업종이면 자유롭게 추가·수정하세요.

점수 산출 방식: 리뷰에 등장하는 키워드의 가중치를 모두 더한 값(`raw`)을 아래 기준으로 1~5점으로 변환합니다.

| raw 합계 | 점수 | NPS 분류 |
|---|---|---|
| ≥ 3 | 5 | 추천고객 |
| 2 | 4 | 추천고객 |
| −1 ~ 1 | 3 | 중립고객 |
| −2 | 2 | 비추천고객 |
| ≤ −3 | 1 | 비추천고객 |

**방법 B. Gemini API 기반 (`llm`)** — 문맥과 부정어까지 이해해 더 정확합니다. [Google AI Studio](https://aistudio.google.com/app/apikey)에서 신용카드 없이 API 키를 발급받으면, `gemini-3.1-flash-lite`(2026년 5월부터 정식 버전, 무료 티어 제공)를 무료 한도 내에서 쓸 수 있습니다. 정확한 분당/일일 한도는 계정마다 달라 [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)의 rate-limit 페이지에서 확인하는 게 정확합니다. 별점이 이미 있는 리뷰는 점수 추정에는 쓰지 않고 핵심 키워드 추출에만 사용합니다.

**LLM 프롬프트도 방법 A와 같은 기준을 공유합니다.** `KEYWORD_DB` 사전(카테고리·가중치·긍/부정 예시 단어)을 그대로 프롬프트에 참고 자료로 주입하고, 점수 1~5점마다 구체적인 예시 문장과 few-shot 입출력 예시 5개를 포함시켜, 키워드 방식만큼 세밀하고 근거가 명확한 판단을 하도록 강화했습니다. 또한 리뷰를 한 건씩 호출하지 않고 `LLM_BATCH_SIZE`(2단계 설정)개씩 묶어 한 번의 API 호출로 함께 채점해, 리뷰가 많을 때도 훨씬 빠르게 끝납니다.

In [ ]:
# ================================================================
# 카테고리별 감성 키워드 사전 (세분화 버전)
# ----------------------------------------------------------------
# - 가중치 범위: -3(매우 부정) ~ +3(매우 긍정). 0은 "점수엔 영향 없이 태그만 남기고 싶은 단어"용.
# - 카테고리(맛/가격/양/포장/배송/서비스/재구매의사/전체만족도)로 나눠두면,
#   각 리뷰가 "왜" 그 점수를 받았는지 바로 보여서 인사이트를 뽑기 훨씬 좋습니다.
# - 아래는 식품/델리 브랜드 리뷰에서 자주 나오는 표현을 폭넓게 반영한 예시입니다.
#   실제 업종/브랜드에 맞게 자유롭게 추가·삭제·수정하세요.
# ================================================================

KEYWORD_DB = {
    "맛/품질": {
        "인생템": 3, "존맛": 3, "찐맛": 3, "최고": 3,
        "맛있": 2, "고소": 2, "풍미": 2, "진한 맛": 2, "깊은 맛": 2, "감칠맛": 2,
        "쫄깃": 1, "쫀득": 1, "촉촉": 1, "부드럽": 1, "신선": 2, "두툼": 1, "담백": 1,
        "괜찮": 1, "무난": 1, "먹을만": 1, "나쁘지 않": 1,
        "심심": -1, "밍밍": -1, "아쉽": -1, "그닥": -1, "그저 그렇": -1,
        "느끼": -2, "짜": -2, "딱딱": -2, "질기": -2, "비리": -2, "퍽퍽": -2, "니글": -2,
        "맛없": -2, "최악": -3, "실망": -3, "역하": -3, "상했": -3, "쉰내": -3, "곰팡이": -3,
    },
    "가격": {
        "가성비": 2, "가성비 갑": 3, "합리적": 2, "저렴": 1,
        "비싸": -1, "가격대 있": -1, "가격 부담": -1,
        "바가지": -3, "돈 아까": -2,
    },
    "양/구성": {
        "양 많": 1, "푸짐": 2, "든든": 1, "넉넉": 1,
        "양 적": -1, "부실": -2, "부족": -1, "생각보다 적": -1,
    },
    "포장": {
        "포장 꼼꼼": 1, "포장 좋": 1, "밀봉 잘": 1, "포장 깔끔": 1,
        "포장 파손": -2, "터짐": -2, "샜": -2, "찌그러": -1, "포장 부실": -2,
    },
    "배송": {
        "배송 빠르": 1, "빠른 배송": 1, "당일 도착": 1, "총알 배송": 2,
        "배송 늦": -1, "배송 지연": -2, "누락": -2, "배송 오류": -2,
    },
    "서비스/CS": {
        "친절": 2, "응대 좋": 1, "빠른 응대": 1,
        "응대 느리": -1, "환불 거부": -3, "연락 두절": -3, "무성의": -2,
    },
    "재구매/추천": {
        "강추": 3, "적극 추천": 3, "재재구매": 2, "재구매": 2, "또 구매": 2, "또 시키": 2, "단골": 2,
        "비추": -2, "다신 안": -3, "절대 비추": -3, "안 살": -2,
    },
    "전체 만족도": {
        "만족": 2, "대만족": 3, "좋아요": 1, "좋았": 1, "행복": 1,
        "안좋": -1, "화나": -2, "짜증": -2, "속상": -1,
    },
    "브랜드 태그(점수 미반영)": {
        "치즈": 0,  # 예시 — 자사 핵심 상품명 등, 점수에는 영향 없이 키워드로만 잡고 싶은 단어
    },
}

# ----------------------------------------------------------------
# 부정어 처리 (간단한 규칙 기반 — 완벽하진 않지만 없는 것보다 훨씬 정확해집니다)
#   1) 접두사형: "불친절", "무성의", "비호감" 처럼 키워드 바로 앞에 공백 없이 부정 접두사가
#      붙으면 반전. "너무"처럼 다른 단어 끝이 우연히 접두사와 같은 글자인 경우는 공백이
#      끼어 있으므로 반전되지 않습니다.
#   2) "안"  : "안 비싸요"처럼 띄어 쓴 부정 표현이 매우 흔해서, 공백 하나를 사이에 둔
#      경우까지 별도로 인정합니다.
#   3) 후위형: "친절하지 않았어요"처럼 키워드 뒤에 "~하지 않다/못하다"가 바로 이어지면 반전
# ----------------------------------------------------------------
NEGATION_ATTACHED_PREFIXES = ("불", "무", "비")
NEGATION_SUFFIXES = ("하지 않", "하지않", "치 않", "치않", "하지 못", "하지못", "치 못", "치못")

ALL_KEYWORDS = {kw: w for cat in KEYWORD_DB.values() for kw, w in cat.items()}
KEYWORD_TO_CATEGORY = {kw: cat for cat, kws in KEYWORD_DB.items() for kw in kws}


def _is_negated(text: str, idx: int, kw_len: int, window: int = 6) -> bool:
    if idx >= 1 and text[idx - 1] in NEGATION_ATTACHED_PREFIXES:
        return True
    if idx >= 2 and text[idx - 1] == " " and text[idx - 2] == "안":
        return True
    suffix_ctx = text[idx + kw_len: idx + kw_len + window]
    return any(suf in suffix_ctx for suf in NEGATION_SUFFIXES)


def score_review_keyword(text: str):
    matched = []
    raw = 0
    for kw, w in ALL_KEYWORDS.items():
        idx = text.find(kw)
        if idx == -1:
            continue
        w_eff = -w if (w != 0 and _is_negated(text, idx, len(kw))) else w
        matched.append((kw, w_eff))
        raw += w_eff

    # 키워드 사전을 세분화한 만큼 raw 합계의 범위도 넓어져서, 5단계 기준을 아래처럼 재조정했습니다.
    if raw >= 5:
        score = 5
    elif 2 <= raw <= 4:
        score = 4
    elif -1 <= raw <= 1:
        score = 3
    elif -4 <= raw <= -2:
        score = 2
    else:
        score = 1

    if matched:
        keywords = ", ".join(f"{KEYWORD_TO_CATEGORY[kw]}:{kw}({w:+d})" for kw, w in matched)
    else:
        keywords = "-"
    return score, keywords


# NPS 분류: 4~5점 = 추천고객, 3점 = 중립고객, 1~2점 = 비추천고객
def classify_nps(score: int) -> str:
    if score >= 4:
        return "추천고객"
    elif score == 3:
        return "중립고객"
    else:
        return "비추천고객"


In [ ]:
import os
def get_gemini_client():
    # aistudio.google.com/app/apikey 에서 신용카드 없이 무료로 API 키 발급
    api_key = os.environ["GOOGLE_API_KEY"]
    from google import genai
    return genai.Client(api_key=api_key)


# ----------------------------------------------------------------
# 키워드 사전(KEYWORD_DB)을 LLM 프롬프트용 참고 자료로 변환
# — 키워드 채점기와 LLM 채점기가 "같은 기준"을 보고 판단하도록,
#   방법 A에서 쓰는 사전을 그대로 재사용합니다. KEYWORD_DB를 수정하면
#   이 참고 자료도 자동으로 함께 갱신됩니다.
# ----------------------------------------------------------------
def build_keyword_reference_block() -> str:
    lines = []
    for cat, kws in KEYWORD_DB.items():
        if cat == "브랜드 태그(점수 미반영)":
            continue
        pos = sorted((k for k, w in kws.items() if w > 0), key=lambda k: -kws[k])
        neg = sorted((k for k, w in kws.items() if w < 0), key=lambda k: kws[k])
        parts = []
        if pos:
            parts.append("긍정: " + ", ".join(pos))
        if neg:
            parts.append("부정: " + ", ".join(neg))
        if parts:
            lines.append(f"- {cat} \u2192 " + " / ".join(parts))
    return "\n".join(lines)


KEYWORD_REFERENCE_BLOCK = build_keyword_reference_block()


# ----------------------------------------------------------------
# 배치용 프롬프트 — 리뷰 여러 건을 한 번에 평가 요청합니다.
# (리뷰 1건마다 API를 호출하는 대신, LLM_BATCH_SIZE건씩 묶어서 호출 횟수 자체를 줄입니다)
# ----------------------------------------------------------------
SENTIMENT_PROMPT_BATCH = """당신은 이 브랜드 리뷰만 수백~수천 건씩 직접 라벨링해온 시니어 CX 데이터 분석가입니다.
아래에 번호가 매겨진 고객 리뷰 목록이 주어집니다. 각 리뷰를 사람이 한 건 한 건 세밀하게 검토하듯
1~5점 척도로 평가하세요. 대충 훑고 점수를 매기지 말고, 아래 참고 자료와 기준을 실제로 하나씩 대조해서 판단하세요.

[참고 — 이 브랜드 리뷰에서 실제로 자주 쓰이는 카테고리별 표현 사전]
(정량 채점기(키워드 방식)가 쓰는 사전과 동일합니다. 리뷰에 아래와 정확히 같은 단어가 없어도,
같은 의미·같은 어감의 다른 표현이면 동일한 카테고리·감성으로 취급하세요.)
{keyword_reference}

[1~5점 평가 기준 — 점수마다 실제로 나올 법한 예시와 함께 판단하세요]

5점 (매우 긍정):
  - "인생템", "존맛", "찐맛", "강추", "적극 추천", "가성비 갑", "총알 배송"처럼 강한 긍정 표현이 하나 이상 있음
  - 맛/가격/양/포장/배송/서비스 중 2개 이상 항목에서 동시에 긍정적임
  - 재구매·추천 의사가 "재구매", "또 시키", "단골"처럼 명확하게 드러남
  - 예: "여기 진짜 인생템이에요 배송도 총알같이 빨라서 완전 만족, 무조건 재구매 각!"

4점 (긍정, 사소한 아쉬움 한두 가지 동반):
  - "맛있", "만족", "좋았" 등 긍정 표현이 우세하지만, "가격대 있", "배송 늦"처럼 사소한 불만 1가지가 함께 있음
  - 예: "맛은 진짜 고소하고 좋은데 가격이 조금 부담스럽긴 해요"

3점 (중립):
  - 긍정·부정 표현이 비슷한 비중으로 섞여 상쇄됨 (서로 다른 항목에서 팽팽하게 맞섬)
  - 또는 감정 표현 없이 배송 확인·구매 사실 전달 등 사실 나열만 있음
  - 예: "잘 받았습니다. 아직 안 먹어봐서 맛은 모르겠어요"

2점 (부정, 격한 표현 없음):
  - "심심", "밍밍", "아쉽", "배송 늦", "부실"처럼 불만은 있지만 "최악", "환불" 수준의 격한 표현은 없음
  - 예: "맛은 그냥 무난한데 배송이 너무 늦게 왔어요"

1점 (매우 부정):
  - "최악", "환불 거부", "다신 안 시켜요", "상했", "곰팡이", "연락 두절"처럼 강한 불만 표현이 있음
  - 또는 맛/배송/서비스 등 여러 항목에서 동시에 부정적이며 강하게 비추천함
  - 예: "곰팡이 핀 거 보내놓고 환불도 안 해줘요 최악입니다 다신 안 삽니다"

[특히 주의해서 판단할 것 — 아래를 놓치면 채점이 틀어집니다]
1. 부정어·반어법: "불친절하지 않았어요"=긍정, "친절하지 않았어요"=부정. "안 비싸요", "나쁘지 않아요"처럼
   부정어가 겹쳐 결과적으로 긍정인 문장에 유의하세요.
2. 강조·반복 표현: ㅎㅎ, ㅋㅋㅋ, ㅜㅜ, !!!, 물결(~), 이모지 등은 감정 강도를 높이는 단서로 활용하세요
   (긍정 문맥의 ㅋㅋㅋ은 5점 쪽, 부정 문맥의 ㅜㅜ는 1~2점 쪽으로 기울 수 있습니다).
3. 복합 리뷰: 한 리뷰 안에 맛/가격/양/포장/배송/서비스/재구매의사 등 여러 항목이 섞여 있으면,
   항목별로 먼저 긍정/부정을 나눠 판단한 뒤 종합해서 최종 점수 하나로 합치세요.
4. 브랜드 고유명사(상품명 등)는 그 자체로는 점수에 영향을 주지 않는 단순 태그이니,
   감정이 실려 있을 때만 점수에 반영하세요.
5. 비교 리뷰("A보다 여기가 낫다")는 지금 평가 대상 브랜드에 대해 이 리뷰가 최종적으로
   내리는 감정을 기준으로 판단하세요.
6. 아래 리뷰들은 서로 다른 손님이 쓴 완전히 독립된 리뷰입니다. 앞뒤 리뷰의 내용에 영향받지 말고
   각 리뷰를 그 리뷰 하나의 내용만으로 개별 판단하세요.

[핵심 키워드 추출 — 키워드 채점기와 같은 카테고리 체계 사용]
각 리뷰마다 점수 판단의 근거가 된 표현을 최소 2개, 최대 10개까지(문장이 짧아 표현이 적으면 있는 만큼만) 뽑고,
각 표현마다 아래 8개 카테고리 중 하나를 붙이고 긍정/부정(또는 중립)도 함께 표시하세요.
카테고리: 맛/품질, 가격, 양/구성, 포장, 배송, 서비스/CS, 재구매/추천, 전체 만족도

[개별 리뷰 평가 예시 — 아래는 하나의 리뷰가 어떻게 채점되는지 보여주는 감각용 예시입니다]

리뷰: "인생 치즈 찾았어요 진짜 존맛인데 배송도 총알 배송이라 완전 만족스러웠습니다 다음에 또 시킬게요"
평가: {{"score": 5, "keywords": ["맛/품질:존맛(긍정)", "배송:총알 배송(긍정)", "전체 만족도:만족스러(긍정)", "재구매/추천:또 시킬게요(긍정)"]}}

리뷰: "맛은 고소하고 좋았는데 양이 생각보다 적어서 좀 아쉬웠어요"
평가: {{"score": 4, "keywords": ["맛/품질:고소하고 좋았(긍정)", "양/구성:생각보다 적(부정)"]}}

리뷰: "그냥 그저 그런 맛이었고 배송은 예정대로 왔습니다"
평가: {{"score": 3, "keywords": ["맛/품질:그저 그런(부정)", "배송:예정대로(중립)"]}}

리뷰: "포장이 터져서 왔고 맛도 좀 짜네요"
평가: {{"score": 2, "keywords": ["포장:터져서(부정)", "맛/품질:짜(부정)"]}}

리뷰: "곰팡이 핀 상품이 와서 환불 요청했는데 연락도 안 받아요 최악입니다"
평가: {{"score": 1, "keywords": ["맛/품질:곰팡이(부정)", "서비스/CS:연락도 안 받아요(부정)", "전체 만족도:최악(부정)"]}}

[응답 형식 — 반드시 지킬 것]
- 아래 목록에 있는 리뷰 {n}개를 번호 순서대로 각각 평가하고, 정확히 {n}개의 JSON 객체로 이루어진 하나의
  JSON 배열로 반환하세요. 배열의 i번째 객체는 반드시 목록의 i번째 리뷰에 대한 평가여야 합니다(순서 유지 필수).
- 각 객체는 {{"score": <1~5 정수>, "keywords": [...]}} 형식이어야 합니다.
- 그 외 설명, 리뷰 번호/원문 재출력, 줄바꿈 텍스트, 마크다운(```json 등)은 절대 포함하지 마세요.

[배치 응답 형식 예시 — 리뷰가 2개인 경우]
입력:
1. "인생 치즈 찾았어요 진짜 존맛인데 배송도 총알 배송이라 완전 만족스러웠습니다 다음에 또 시킬게요"
2. "곰팡이 핀 상품이 와서 환불 요청했는데 연락도 안 받아요 최악입니다"
출력:
[{{"score": 5, "keywords": ["맛/품질:존맛(긍정)", "배송:총알 배송(긍정)", "재구매/추천:또 시킬게요(긍정)"]}}, {{"score": 1, "keywords": ["맛/품질:곰팡이(부정)", "서비스/CS:연락도 안 받아요(부정)", "전체 만족도:최악(부정)"]}}]

[지금 평가할 리뷰 목록 — 총 {n}개]
{reviews}
"""


def _build_batch_prompt(review_batch: list[str]) -> str:
    numbered = "\n".join(f'{i + 1}. "{text}"' for i, text in enumerate(review_batch))
    return SENTIMENT_PROMPT_BATCH.format(
        keyword_reference=KEYWORD_REFERENCE_BLOCK,
        n=len(review_batch),
        reviews=numbered,
    )


def score_reviews_llm_batch(client, review_batch: list[str], model: str = "gemini-3.1-flash-lite", retries: int = 3):
    """리뷰 여러 건을 한 번의 API 호출로 채점합니다. review_batch와 같은 길이, 같은 순서의
    [(score, keywords_str), ...] 리스트를 반환합니다.

    응답이 잘리거나(JSON 파싱 실패) 개수가 안 맞으면 재시도하고, 그래도 실패하면 배치를
    절반으로 쪼개 재귀적으로 다시 시도합니다 — 리뷰 한두 건의 문제로 배치 전체가
    중립 처리되는 것을 막기 위함입니다. 리뷰 1건짜리 배치까지 쪼갰는데도 실패하면
    그 리뷰만 중립(3점)으로 처리합니다.
    """
    from google.genai import types
    n = len(review_batch)
    prompt = _build_batch_prompt(review_batch)
    max_tokens = min(32768, 250 * n + 400)  # 배치가 커질수록 응답도 길어지므로 건수에 비례해 여유를 둠

    for attempt in range(retries):
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",  # JSON 형식으로 응답을 강제
                    temperature=0.2,
                    max_output_tokens=max_tokens,
                ),
            )
            data = json.loads(resp.text)
            if not isinstance(data, list) or len(data) != n:
                got = len(data) if isinstance(data, list) else type(data).__name__
                raise ValueError(f"리뷰 {n}건을 요청했는데 응답은 {got}개 — 개수 불일치(응답이 잘렸을 가능성)")
            return [(int(d["score"]), ", ".join(d.get("keywords", []))) for d in data]
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # 요청 한도 초과 등 일시적 오류 대비 지수 백오프 재시도
                continue
            if n > 1:
                mid = n // 2
                print(f"  \u26a0\ufe0f 배치({n}건) 처리 실패, 절반으로 나눠 재시도: {e}")
                return (score_reviews_llm_batch(client, review_batch[:mid], model=model, retries=retries)
                        + score_reviews_llm_batch(client, review_batch[mid:], model=model, retries=retries))
            print(f"  \u26a0\ufe0f 분류 실패, 중립(3점) 처리: {e}")
            return [(3, "-")]

# gemini-3.1-flash-lite는 2026년 5월부터 정식(stable) 버전이며 무료 티어가 제공됩니다.
# 무료 티어의 정확한 분당/일일 요청 한도는 계정마다 다르고 공식 문서에도 고정 수치가 나와 있지 않으니,
# 실행 전 https://aistudio.google.com/app/apikey 의 rate-limit 페이지에서 본인 계정 기준으로 확인하세요.
# 위 함수는 한도 초과·응답 잘림 등 오류 시 자동으로 재시도/배치 분할하며, LLM_BATCH_SIZE(2단계 설정)로
# 한 번에 몇 건씩 묶을지 조절할 수 있습니다.


### 실행 — 전체 리뷰 점수 매기기

In [ ]:
results = []
counts = {"추천고객": 0, "중립고객": 0, "비추천고객": 0}  # 진행 중 실시간 집계용

if SENTIMENT_METHOD == "keyword":
    pbar = tqdm(reviews, desc="점수 산정 중")
    for item in pbar:
        text = item["리뷰"]
        _, kw = score_review_keyword(text)
        if item.get("별점") is not None:
            score, source = int(item["별점"]), "실제 별점"
        else:
            score, source = score_review_keyword(text)[0], "키워드 추정"
        nps_class = classify_nps(score)
        counts[nps_class] += 1
        results.append({"리뷰": text, "점수(1~5)": score, "점수 출처": source, "핵심 키워드": kw, "NPS 분류": nps_class})
        pbar.set_postfix(추천=counts["추천고객"], 중립=counts["중립고객"], 비추천=counts["비추천고객"])

elif SENTIMENT_METHOD == "llm":
    client = get_gemini_client()
    review_texts = [item["리뷰"] for item in reviews]

    # 리뷰 전체를 LLM_BATCH_SIZE 단위로 잘라 배치 호출합니다.
    # 배치가 남아있는 동안 계속 반복하고, 리뷰를 전부 처리해 더 이상 배치가 없으면
    # 자연스럽게 아래로 빠져나가 5단계(CSV 저장)로 이어집니다.
    batches = [list(range(i, min(i + LLM_BATCH_SIZE, len(review_texts))))
               for i in range(0, len(review_texts), LLM_BATCH_SIZE)]

    pbar = tqdm(batches, desc=f"Gemini 배치 분석 중 (배치당 최대 {LLM_BATCH_SIZE}건, 총 {len(batches)}배치)")
    for idx_batch in pbar:
        batch_texts = [review_texts[i] for i in idx_batch]
        batch_results = score_reviews_llm_batch(client, batch_texts)

        for i, (llm_score, kw) in zip(idx_batch, batch_results):
            item = reviews[i]
            if item.get("별점") is not None:
                score, source = int(item["별점"]), "실제 별점"
            else:
                score, source = llm_score, "Gemini 추정"
            nps_class = classify_nps(score)
            counts[nps_class] += 1
            results.append({"리뷰": item["리뷰"], "점수(1~5)": score, "점수 출처": source, "핵심 키워드": kw, "NPS 분류": nps_class})

        pbar.set_postfix(추천=counts["추천고객"], 중립=counts["중립고객"], 비추천=counts["비추천고객"])
        time.sleep(0.5)  # 배치 사이 요청 속도 조절 (계정별 무료 티어 한도가 다르니 429 오류가 잦으면 늘리세요)

else:
    raise ValueError("SENTIMENT_METHOD는 'keyword' 또는 'llm' 이어야 합니다.")

df = pd.DataFrame(results)
print(f"\n✅ 총 {len(df)}건 점수 산정 완료 — 추천고객 {counts['추천고객']}건 / 중립고객 {counts['중립고객']}건 / 비추천고객 {counts['비추천고객']}건")
print(f"(아래는 미리보기로 처음 10건만 표시됩니다. 전체 {len(df)}건은 df 변수와 6단계 CSV에 모두 들어있습니다)")
df.head(10)


## 5. NPS 점수 계산 및 요약표

In [ ]:
summary = df["NPS 분류"].value_counts().reindex(["추천고객", "중립고객", "비추천고객"]).fillna(0).astype(int)
total = int(summary.sum())
summary_pct = (summary / total * 100).round(1)

nps_score = summary_pct["추천고객"] - summary_pct["비추천고객"]

summary_table = pd.DataFrame({"건수": summary, "비율(%)": summary_pct})
summary_table.loc["합계"] = [total, 100.0]

print(f"\U0001F4CA {BRAND_NAME} 소셜 NPS 점수: {nps_score:.1f}점  (추천고객 비율 − 비추천고객 비율)")
print(f"   추천고객 {summary['추천고객']}건({summary_pct['추천고객']}%) · "
      f"중립고객 {summary['중립고객']}건({summary_pct['중립고객']}%) · "
      f"비추천고객 {summary['비추천고객']}건({summary_pct['비추천고객']}%)  |  전체 {total}건")

# 표를 더 크고 굵은 글씨로 보기 좋게 스타일링
styled_summary = (
    summary_table.style
    .set_caption(f"{BRAND_NAME} NPS 요약표")
    .format({"건수": "{:.0f}", "비율(%)": "{:.1f}"})
    .set_properties(**{"font-size": "16px", "font-weight": "bold", "text-align": "center", "padding": "8px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "20px"), ("font-weight", "bold"),
                                           ("text-align", "center"), ("padding-bottom", "8px")]},
        {"selector": "th", "props": [("font-size", "16px"), ("font-weight", "bold"),
                                      ("background-color", "#f1f3f4"), ("padding", "8px")]},
    ])
)
styled_summary


## 6. 결과 저장 (CSV)

In [ ]:
safe_name = BRAND_NAME.replace(" ", "_")
detail_csv = f"{safe_name}_NPS_분석결과.csv"
summary_csv = f"{safe_name}_NPS_요약표.csv"

# df에는 화면에 미리보기(head)로 보였던 것과 무관하게 수집·채점된 전체 리뷰가 다 들어있습니다.
df.to_csv(detail_csv, index=False, encoding="utf-8-sig")     # utf-8-sig: 엑셀에서 열 때 한글 깨짐 방지
summary_table.to_csv(summary_csv, encoding="utf-8-sig")

print(f"저장 완료 → {detail_csv} (리뷰 {len(df)}건), {summary_csv} (분류 {len(summary_table) - 1}종 + 합계)")
print("※ 점수 매기기 전 원본은 4단계 앞의 '원본수집리뷰.csv'에 이미 별도 저장되어 있습니다.")


## 7. 그래프 시각화 (총 5종)

나눔고딕 폰트, 큰 글씨, 볼드체가 모든 그래프에 공통 적용됩니다 (1-1단계에서 설정).

1. NPS 분류별 건수 막대그래프 + 비율 파이차트
2. 점수(1~5) 분포 막대그래프
3. 핵심 키워드 Top 15 빈도 가로 막대그래프
4. 점수 출처(실제 별점 vs 추정)별 NPS 분류 비교
5. 카테고리(맛/가격/배송 등)별 언급 빈도


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {"추천고객": "#1a73e8", "중립고객": "#9aa0a6", "비추천고객": "#d93025"}

# (1) 분류별 건수 막대그래프
plot_summary = summary  # 추천고객/중립고객/비추천고객만 (합계 제외)
bars = axes[0].bar(plot_summary.index, plot_summary.values,
                    color=[colors[c] for c in plot_summary.index],
                    edgecolor="black", linewidth=1.2)
axes[0].set_title(f"{BRAND_NAME} 리뷰 NPS 분류 건수", fontweight="bold")
axes[0].set_ylabel("건수", fontweight="bold")
for b in bars:
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height(), int(b.get_height()),
                 ha="center", va="bottom", fontsize=15, fontweight="bold")

# (2) 분류별 비율 파이차트
wedges, texts, autotexts = axes[1].pie(
    plot_summary.values, labels=plot_summary.index, autopct="%1.1f%%",
    colors=[colors[c] for c in plot_summary.index], startangle=90,
    wedgeprops={"edgecolor": "black", "linewidth": 1.2},
    textprops={"fontsize": 14, "fontweight": "bold"},
)
for at in autotexts:
    at.set_fontsize(14)
    at.set_fontweight("bold")
    at.set_color("white")
axes[1].set_title(f"{BRAND_NAME} NPS 비율", fontweight="bold")

plt.suptitle(f"{BRAND_NAME} 소셜 NPS 점수: {nps_score:.1f}점", fontsize=22, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{safe_name}_NPS_그래프.png", dpi=150)
plt.show()


In [ ]:
# 점수(1~5) 분포도 — 어느 점수대에 리뷰가 몰려있는지 확인
plt.figure(figsize=(8, 5))
score_counts = df["점수(1~5)"].value_counts().sort_index()
bars = plt.bar(score_counts.index.astype(str), score_counts.values, color="#4c8bf5",
               edgecolor="black", linewidth=1.2)
plt.title(f"{BRAND_NAME} 리뷰 점수(1~5) 분포", fontweight="bold")
plt.xlabel("점수", fontweight="bold")
plt.ylabel("리뷰 수", fontweight="bold")
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height(), int(b.get_height()),
              ha="center", va="bottom", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{safe_name}_점수분포.png", dpi=150)
plt.show()


In [ ]:
# 핵심 키워드 Top 15 빈도 — 어떤 표현이 가장 많이 언급됐는지 한눈에 확인
from collections import Counter


def parse_keyword_field(field: str):
    """'카테고리:키워드(+2), 카테고리:키워드(-1)' 형식에서 키워드 이름만 뽑아냅니다."""
    if not isinstance(field, str) or field == "-":
        return []
    names = []
    for part in field.split(", "):
        part = part.strip()
        if ":" in part:
            part = part.split(":", 1)[1]
        name = re.sub(r"\(.*?\)", "", part).strip()
        if name:
            names.append(name)
    return names


keyword_counter = Counter()
for field in df["핵심 키워드"]:
    keyword_counter.update(parse_keyword_field(field))

top_keywords = keyword_counter.most_common(15)
if top_keywords:
    names, freqs = zip(*top_keywords)
    plt.figure(figsize=(10, 8))
    bars = plt.barh(names[::-1], freqs[::-1], color="#1a73e8", edgecolor="black", linewidth=1.0)
    plt.title(f"{BRAND_NAME} 핵심 키워드 Top {len(top_keywords)}", fontweight="bold")
    plt.xlabel("등장 횟수", fontweight="bold")
    for b in bars:
        plt.text(b.get_width() + 0.15, b.get_y() + b.get_height() / 2, int(b.get_width()),
                  va="center", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{safe_name}_핵심키워드_TOP15.png", dpi=150)
    plt.show()
else:
    print("매칭된 키워드가 없어 키워드 빈도 그래프를 건너뜁니다.")


In [ ]:
# 점수 출처(실제 별점 vs 추정)별 NPS 분류 비교 — 실제 별점과 추정치의 경향이 다른지 확인
source_summary = pd.crosstab(df["점수 출처"], df["NPS 분류"])
source_summary = source_summary.reindex(columns=["추천고객", "중립고객", "비추천고객"], fill_value=0)

ax = source_summary.plot(kind="bar", stacked=True, figsize=(9, 6),
                          color=[colors["추천고객"], colors["중립고객"], colors["비추천고객"]],
                          edgecolor="black", linewidth=1.0)
plt.title(f"{BRAND_NAME} 점수 출처별 NPS 분류 비교", fontweight="bold")
plt.xlabel("점수 출처", fontweight="bold")
plt.ylabel("건수", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(title="NPS 분류", title_fontsize=12, fontsize=12)
plt.tight_layout()
plt.savefig(f"{safe_name}_점수출처별_비교.png", dpi=150)
plt.show()


In [ ]:
# 카테고리(맛/가격/배송 등)별 언급 빈도 — 고객이 어떤 측면을 가장 많이 이야기하는지 확인
from collections import Counter

category_counter = Counter()
for field in df["핵심 키워드"]:
    if not isinstance(field, str) or field == "-":
        continue
    for part in field.split(", "):
        part = part.strip()
        if ":" in part:
            category_counter[part.split(":", 1)[0].strip()] += 1

if category_counter:
    cats, freqs = zip(*category_counter.most_common())
    plt.figure(figsize=(9, 6))
    bars = plt.bar(cats, freqs, color="#34a853", edgecolor="black", linewidth=1.0)
    plt.title(f"{BRAND_NAME} 리뷰 카테고리별 언급 빈도", fontweight="bold")
    plt.ylabel("언급 횟수", fontweight="bold")
    plt.xticks(rotation=30, ha="right")
    for b in bars:
        plt.text(b.get_x() + b.get_width() / 2, b.get_height(), int(b.get_height()),
                  ha="center", va="bottom", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{safe_name}_카테고리별_언급빈도.png", dpi=150)
    plt.show()
else:
    print("카테고리 정보가 없어 카테고리별 그래프를 건너뜁니다.")


## 8. 확장 — 여러 브랜드/링크 비교하기 (선택)

여러 브랜드를 한 번에 비교하고 싶다면, 위 2~5단계를 함수로 감싸 아래처럼 반복 실행하면 됩니다.

In [ ]:
# def run_pipeline(url: str, brand: str, sentiment_method="keyword", max_pages=500, max_items=None) -> dict:
#     items = collect_reviews_auto(url, max_pages=max_pages, max_items=max_items)
#     rows = []
#     for item in items:
#         _, kw = score_review_keyword(item["리뷰"])
#         score = int(item["별점"]) if item.get("별점") is not None else score_review_keyword(item["리뷰"])[0]
#         rows.append({"리뷰": item["리뷰"], "점수(1~5)": score, "핵심 키워드": kw})
#     d = pd.DataFrame(rows)
#     d["NPS 분류"] = d["점수(1~5)"].apply(classify_nps)
#     pct = d["NPS 분류"].value_counts(normalize=True).reindex(["추천고객", "중립고객", "비추천고객"]).fillna(0) * 100
#     return {"브랜드": brand, "NPS": round(pct["추천고객"] - pct["비추천고객"], 1), "리뷰수": len(d)}
#
# BRAND_URLS = {
#     "브랜드A": "https://example.com/a/board/product/list.html?board_no=4",
#     "브랜드B": "https://example.com/b/board/product/list.html?board_no=4",
# }
# compare = pd.DataFrame([run_pipeline(u, b) for b, u in BRAND_URLS.items()])
#
# plt.figure(figsize=(6, 4))
# plt.bar(compare["브랜드"], compare["NPS"], color="#1a73e8")
# plt.title("브랜드별 소셜 NPS 비교")
# plt.ylabel("NPS 점수")
# plt.tight_layout()
# plt.show()


---
### 참고 / 한계
- **robots.txt는 코드가 매 페이지 요청 전에 자동으로 확인합니다.** 1페이지부터 시작해 마지막 페이지까지 계속 순회하며 수집하다가, 목록 URL은 허용하면서 `&page=` 파라미터만 차단하는 사이트를 만나면(예시 링크가 그렇습니다) 그 지점에서 멈추고 이유를 출력합니다 — 조용히 건너뛰지 않습니다. 이는 사이트가 스스로 정한 크롤링 정책이므로 코드가 이를 우회하지는 않습니다.
- **리뷰 수와 무관하게 기본적으로 끝까지 다 수집합니다.** `MAX_PAGES`(2단계 설정)는 이상 동작으로 인한 무한 루프만 막는 안전장치이고, 실제로는 "새 리뷰 없음" 또는 "robots.txt 차단"을 만나는 순간 자동으로 멈춥니다.
- **점수는 실제 별점이 있으면 그것을 그대로 쓰고, 없을 때만 추정합니다.** `df["점수 출처"]` 컬럼으로 각 행의 점수가 실제 별점인지 추정치인지 항상 확인할 수 있습니다.
- 키워드 사전(`POSITIVE_KEYWORDS`, `NEGATIVE_KEYWORDS`, `EXTRA_KEYWORDS`)은 업종/브랜드에 맞게 계속 다듬을수록 정확도가 올라갑니다.
- 대량(수백~수천 건) 리뷰를 `llm` 모드로 분류할 때 429(요청 한도 초과) 오류가 잦다면 `LLM_BATCH_SIZE`(2단계 설정)를 줄이거나 `score_reviews_llm_batch`의 `time.sleep` 간격을 늘리세요. 계정별 정확한 무료 한도는 [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)의 rate-limit 페이지에서 확인할 수 있습니다.
- 자동 수집이 robots.txt로 막힌 사이트는 `COLLECT_MODE = "manual"`로 바꿔, 관리자 페이지 내보내기나 직접 복사한 리뷰를 CSV(`리뷰` 컬럼)로 올려서 그대로 4단계부터 이어가면 됩니다.
